The aim of this project is to implement Gadient Descent as part of studying Machine Learning.
We will implemenet the basic algorithm, consider different step sizes and iterations, 
see the effect of a regularizer, and compare to $\texttt{sklearn}$. We use $\texttt{sklearn}$'s
California Housing data as our test case.

In [6]:
from sklearn.datasets import fetch_california_housing as cali        
import numpy
import pandas as pd

In [2]:
data = cali()
X = data.data 
Y = data.target
print(X.shape)
print(Y.shape)

(20640, 8)
(20640,)


In [3]:
# now we must shhuffle data. To keep the features of each datum together,
# we use indices to access the feauture vectors

n = len(X)
indices = numpy.arange(n) # use numpy arange to be consistent with numpy
# and because range doesn't create an actual array that can be shuffled
# think or numpy.arange() as array-range not the word "arranging"
numpy.random.shuffle(indices)

# now we do the train/test split
split = int(.8 * n) # make sure to cast to int
train_idx = indices[:split]
test_idx = indices[split:]
X_train, X_test = X[train_idx], X[test_idx] # fancy indexing: get matrices
# by inexing with arrays ("fancy indexing")
Y_train, Y_test = Y[train_idx], Y[test_idx]

# now normalize everything!
means = numpy.mean(X_train, axis = 0) # axis = 0 means keep the first 
# axis (keep rows take average over column)
stdev = numpy.std(X_train, axis = 0)
X_train_normal = (X_train - means)/stdev # can apply tof each entry row-wise
# like this
X_test_normal = (X_test - means)/stdev
# add a column of ones to both the X train matrix and X test matrix
# using the function hstack
ones_train = numpy.ones((len(X_train_normal), 1))
ones_test= numpy.ones((len(X_test_normal), 1))
X_train_normal = numpy.hstack([ones_train, X_train_normal])
X_test_normal = numpy.hstack([ones_test, X_test_normal])


We create the following functions:
1. $\texttt{residuals}$ calculates the residuals of the data:
\begin{gather*}
\begin{bmatrix} y_{1} \\ y_{2}\\ y_{3} \\ \vdots \\y_{N} \end{bmatrix}
= X \begin{bmatrix} w_{1} \\ w_{2}\\ w_{3} \\ \vdots \\w_{N} \end{bmatrix}
\end{gather*}
where $X$ stores the feature data for each data point and $w$ is our parameter vector. This creates a a vector in which each line
represents one data point's estimate subtracted from its label y.

2. $\texttt{mse\_loss}$ calculates the average of the sqaured residuals over all the data points.
\begin{gather*}
\frac{1}{N}\sum_{i = 1}^{N} r_{i}^2
\end{gather*}
where $N$ is the size of the data set and $r_{i}$ is the $i^{\text{th}}$ residual, the $i^{\text{th}}$
row of the vector outputted by $\texttt{residuals}$.

3. $\texttt{lossFunction}$ is the same as $\texttt{mse\_loss}$ except that we add the regularization term
$ \lambda || w ||^{2}$.

4. $\texttt{gradient}$: For this we compute the gradient of the regularized loss function 
\begin{gather*}
l^*(w) = \frac{1}{N}\sum_{i = 1}^N r_{i}^2 + \lambda(w_1^2 + \cdots + w_{k}^2)
\\ \implies \frac{\partial l^*}{\partial w_j} = \frac{1}{N}\sum_{i = 1}^N 2r_i(-x_{ij}) + 2\lambda w_{j}, j \in {1, \ldots, k}\\
\frac{\partial l^*}{\partial w_0} = -\frac{1}{N}\sum_{i = 1}^N 2r_{i}
\end{gather*}
We can calculate the first term $-\frac{1}{N}\sum_{i = 1}^N 2r_{i}x_{ij}$ with $-X^{T} r$ with normal matrix multiplication ($\texttt{dot}$ in $\texttt{numpy}$). This also covers the special case for $\frac{\partial l}{\partial w_0}$ since we set the entire first column of $X$ to have only ones. For the gradient of the regularization term, we manually deal with the special case for the intercept parameter by manually setting it to zero.

5. $\texttt{GD}$. Here we implement the Gradient Descent algorithm. We set the parameters to an inital setting of zero (since we normalized them this should be the mean) and then we iterate
\begin{gather*}
w_{l+1} = w_{l} - \eta \nabla l^*(w)
\end{gather*}

In [25]:
def residuals(param, X_matrix, Y_vector):
    return Y_vector - numpy.dot(X_matrix, param)

def mse_loss(param, X_matrix, Y_vector):
    res = residuals(param, X_matrix, Y_vector)
    return numpy.sum(res **2 )/len(X_matrix)

def lossFunction(param, X_matrix, Y_vector, gimmel = 0):
    res = residuals(param, X_matrix, Y_vector)
    return numpy.sum(res **2 )/len(X_matrix) + gimmel * numpy.dot(param, param)

def gradient(param, X_matrix, Y_matrix, gimmel = 0):
    reg = 2 * gimmel * param
    reg[0] = 0
    return (-2/len(X_matrix)) * numpy.dot(X_matrix.T, residuals(param, X_matrix, Y_matrix)) + reg

def GD(X_matrix, Y_vector, iter, eta = 0.01, gimmel = 0):
    params = numpy.zeros(X_matrix.shape[1])
    for _ in range(iter):
        params = params - eta * gradient(params, X_matrix, Y_vector, gimmel)
    return params

In [19]:
w = GD(X_train_normal, Y_train, iter=1000, eta=0.01)
print("Train loss:", lossFunction(w, X_train_normal, Y_train))
print("Test loss:", lossFunction(w, X_test_normal, Y_test))

Train loss: 0.5278730628971345
Test loss: 0.5459737124507237


In [20]:
rows = []

def run_experiment(eta, iterations):
    w = GD(X_train_normal, Y_train, iter=iterations, eta=eta)
    train_loss = lossFunction(w, X_train_normal, Y_train)
    test_loss = lossFunction(w, X_test_normal, Y_test)
    gen_error = abs(test_loss - train_loss)
    return {
        'eta': eta,
        'iterations': iterations,
        'train_loss': round(train_loss, 4),
        'test_loss': round(test_loss, 4),
        'generalization_error': round(gen_error, 4)
    }

# run experiments
experiments = [
    (0.001, 1000),
    (0.01, 1000),    # your baseline
    (0.1, 1000),
    (0.01, 100),
    (0.01, 500),
    (0.01, 2000),
]

for eta, iters in experiments:
    rows.append(run_experiment(eta, iters))

results = pd.DataFrame(rows)
print(results)

     eta  iterations  train_loss  test_loss  generalization_error
0  0.001        1000      0.7096     0.7409                0.0313
1  0.010        1000      0.5279     0.5460                0.0181
2  0.100        1000      0.5215     0.5363                0.0147
3  0.010         100      0.7064     0.7375                0.0311
4  0.010         500      0.5497     0.5702                0.0205
5  0.010        2000      0.5219     0.5379                0.0160


In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(X_train_normal, Y_train)
test_loss = mean_squared_error(Y_test, model.predict(X_test_normal))
train_loss = mean_squared_error(Y_train, model.predict(X_train_normal))
print("sklearn test loss:", test_loss)
print("sklearn train loss:", train_loss)
generalization_error = numpy.abs(test_loss - train_loss)
print("Generalization error:", generalization_error)

sklearn test loss: 0.536251279284215
sklearn train loss: 0.5215071977615098
Generalization error: 0.014744081522705232


In [27]:
rows = []

def run_experiment(eta, iterations, gimmel):
    w = GD(X_train_normal, Y_train, iter=iterations, eta=eta, gimmel = gimmel)
    train_loss = mse_loss(w, X_train_normal, Y_train)
    test_loss = mse_loss(w, X_test_normal, Y_test)
    gen_error = abs(test_loss - train_loss)
    return {
        'gimmel': gimmel,
        'train_loss': round(train_loss, 4),
        'test_loss': round(test_loss, 4),
        'generalization_error': round(gen_error, 4)
    }

# run experiments
experiments = [0.5, 0.1, 0.01, 0.001, 0.0001, 0]

for gim in experiments:
    rows.append(run_experiment(0.1, 1000, gim))

results = pd.DataFrame(rows)
print(results)

   gimmel  train_loss  test_loss  generalization_error
0  0.5000      0.6952     0.7147                0.0196
1  0.1000      0.5692     0.5896                0.0204
2  0.0100      0.5234     0.5397                0.0164
3  0.0010      0.5215     0.5365                0.0149
4  0.0001      0.5215     0.5363                0.0148
5  0.0000      0.5215     0.5363                0.0147
